# သင်ခန်းစာ ၀၈ - အမျိုးမျိုးသောနယ်ပယ်မှုဆိုင်ရာ ဒီဇိုင်းပုံစံ


## စီစဉ်တပ်ဆင်ခြင်း


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ဘာကြောင့် Multi-Agent Systems လဲ?

ခရီးစဉ်အစီအစဉ်ပြုလုပ်ခြင်းကဲ့သို့သော အမှန်တကယ်လုပ်ဆောင်ရမည့်တာဝန်များတွင် ဝန်ဆောင်မှုအမျိုးမျိုးရှိသည် — အသင်းလိုဂျစ်စတစ်စ်၊ ဒေသဆိုင်ရာအသိပညာ၊ ဘတ်ဂျက်ချမှတ်ခြင်း နှင့် အခြားများတို့ပါဝင်သည်။ တစ်ဦးတည်းသော agent သည် အရာအားလုံးကို ကိုင်တွယ်ရန်ကြိုးစားသည်မှာ လျင်မြန်စွာ မထိရောက်တော့ပါ။

Multi-agent systems သည် **အထူးပြုခြင်း** ဖြင့် ဤပြဿနာကို ဖြေရှင်းသည်။ agent တစ်ဦးစီသည် အထူးကျွမ်းကျင်သည့် နယ်ပယ်တစ်ခုကို ဦးတည်ကာ ပိုမိုအရည်အသွေးမြင့် ရလဒ်များ ထုတ်ပေးသည်။ ထို့အတူ **တိုးချဲ့နိုင်ခြင်း** ကိုလည်း တိုးတက်စေသည် — အရှိန်လွှတ်ရန် agent အသစ်များ (ဥပမာ၊ လေယာဉ်အထူးကျွမ်းကျင်သူ၊ မ ресторан သုံးသပ်သူ) ကို လွယ်လင့်တကူ ပေါင်းထည့်နိုင်ပြီး ရှိပြီးသား အလုပ်စဉ်ကို ပြန်ရေးရန် မလိုပါ။ agent များကို ဖွဲ့စည်းထားသော လမ်းကြောင်းတစ်ခုဖြင့် ပေါင်းစည်းပြီး ၊ တစ်ယောက်မှ နောက်တစ်ယောက်သို့ အခြေအနေကို ပေးပို့ကြသည်။


## အထူးပြု ကိုယ်စားလှယ်များ ဖန်တီးခြင်း


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Sequencing Workflow တည်ဆောက်ခြင်း

`WorkflowBuilder` သည် နယ်ပယ်တစ်ခုအတွင်း Agents များကိုပစ်ချိတ်စနစ်ဖြင့်တည်ဆောက်နိုင်စေသည်။ ဒီမှာ ရိုးရှင်းတဲ့ နှစ်အဆင့် Pipeline တစ်ခု ပြုလုပ်ထားတယ် - **TravelPlanner** သည် ခရီးစဉ် အကြမ်းဖျဉ်းရေးဆွဲပြီး၊ ထို့နောက် **TravelConcierge** က ဒါကို ပြန်လည်သုံးသပ်ပြီး တိုးတက်ကောင်းမွန်အောင် ပြုပြင်ပေးတယ်။


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## အလုပ်စီးဆင်းမှုတွင် အေးဂျင့်များ ပိုမိုထည့်သွင်းခြင်း

multi-agent ပုံစံ၏ အကြီးစားအားသာချက်တစ်ခုမှာ တိုးချဲ့ရန် မြန်ဆန်လွယ်ကူခြင်းဖြစ်သည်။ အောက်တွင် လေ့လာသူ၏ ဘတ်ဂျက်နှင့် ဇယားကို စစ်ဆေးပြီး၊ ကုန်ကျစရိတ်ကျော်လွန်နိုင်သည့် အချက်များကို သတိပေးကာ ငွေချွေတာနိုင်သော အစီအစဉ်များကို အကြံပြုပေးသည့် **BudgetReviewer** အေးဂျင့်ကို ထည့်သွင်းထားသည်။ ယခု workflow သည် အေးဂျင့်သုံးခုကို ဆက်တိုက် လည်ပတ်သည်- 

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## အကျဉ်းချုပ်

ဤသင်ခန်းစာတွင် သင်လေ့လာသွားသောအချက်များမှာ -

1. **အထူးပြုလုပ်ထားသော အေးဂျင့်များကို ဖန်တီးခြင်း** — အထူးအာရုံစူးစိုက်ချက်ရှိသော အခန်းကဏ္ဍများဖြစ်သော (အစီအစဉ်ရေးဆွဲခြင်း၊ ဝန်ဆောင်မှု၊ ဘတ်ဂျက်သုံးသပ်ခြင်း)။
2. **`WorkflowBuilder` နှင့် `add_edge` ကို အသုံးပြု၍ အေးဂျင့်များကို သွားလာမှုစဉ်လိုက် အလုပ်လုပ်စေခြင်း။**
3. **အေးဂျင့်အများကြီးပါရှိသော လုပ်ဆောင်ခြင်းမှ ထွက်ရှိလာသော အချက်အလက်များကို တစ်စိတ်တစ်ပိုင်းစီ ထွက်ပေါ်လာစဉ်များအား စောင့်ကြည့်ခြင်း။**
4. **ကိန်းချုပ်ထားသော အေးဂျင့်များကို မပြောင်းလဲဘဲ လုပ်ငန်းစဉ်ကို အသစ်သော အေးဂျင့်များဖြင့် တိုးချဲ့ခြင်း။**

အေးဂျင့်အများပါဝင်သော ဒီဇိုင်းပုံစံသည် တစ်ဦးချင်း အေးဂျင့်တစ်ခုစီအား ရိုးရှင်းစွာ ထိန်းသိမ်းထားပြီး၊ တစ်ဦးချင်းအေးဂျင့် တစ်ခုသာ ထုတ်လုပ်နိုင်သည့် ထက် ပိုမိုသာယာပြီး ပြည့်စုံစွာ သုံးသပ်ထားသော ရလဒ်များကို ထုတ်ပေးနိုင်ပါသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**အသိပေးချက်**  
ဤစာရွက်စာတမ်းကို AI ဘာသာပြန်စနစ်ဖြစ်သော [Co-op Translator](https://github.com/Azure/co-op-translator) မှ အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းထားသော်လည်း၊ အလိုအလျောက်ဘာသာပြန်မှုများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ဖြစ်ပေါ်နိုင်ကြောင်း သတိပြုပါရန် မေတ္တာရပ်ခံအပ်ပါသည်။ မူရင်းစာရွက်စာတမ်းကို သူ့ရဲ့ မူလဘာသာဖြင့်သာ ပြဋ္ဌာန်းထားသည့် ထောက်ခံမှုရှိသော မူရင်းလက်ဝါးစာရွက်အဖြစ် ယူဆသင့်ပါသည်။ အရေးကြီးသော အချက်အလက်များအတွက်တော့ ကျွမ်းကျင် လူ့ဘာသာပြန်မှုကို သုံးစွဲရန် အကြံပြုပါသည်။ ဤဘာသာပြန်မှုကို အသုံးပြုရာမှ ဖြစ်ပေါ်လာနိုင်သော နားလည်မှု မှားယွင်းမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
